In [1]:
#| default_exp hyperparameter_tuning

## Hyperparameter Tuning

Peshbeen provides two powerful tools for hyperparameter tuning: `hyperopt_tune` and `optuna_tune`. These functions allow you to optimize the hyperparameters of your forecasting models using the `hyperopt` and `optuna` libraries, respectively. Both functions support cross-validation and can be used with any of the forecasting models available in peshbeen.

### hyperopt_tune example for univariate forecasting using machine learning models

In [2]:
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import hyperopt_tune
from feature_engine.encoding import OneHotEncoder
ohe = OneHotEncoder(drop_last=True)

load_wales_admissions["day_of_week"] = load_wales_admissions.index.dayofweek
load_wales_admissions["month"] = load_wales_admissions.index.month
# split the data into train and test sets
train = load_wales_admissions[:-30]
test = load_wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
ml_model = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)

# Define the hyperparameter search space for LightGBM
from hyperopt import hp
from hyperopt.pyll import scope
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
           'top_rate' : hp.quniform('top_rate', 0.05, 0.4, 0.0001),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'lags': hp.choice("lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14],
                                 [1,2,3,4,5,6,7,14,21],
                                 [1,2,3],
                             ]),
                "seed":0,
                "box_cox": hp.uniform("box_cox", 0.0, 4),
                "box_cox_biasadj": hp.choice("box_cox_biasadj", [True, False])}

# Run hyperparameter tuning using hyperopt
best_params, best_lags, other_ = hyperopt_tune(model=ml_model, df=train, cv_split=5, step_size=10,
                                        test_size=1, eval_metric=RMSE, eval_num=10,
                                        param_space=lgb_param_space)

print("Best params:", best_params)
print("Best lags:", best_lags)
print("Other info:", other_)

100%|██████████| 10/10 [00:22<00:00,  2.28s/trial, best loss: 47.995333574400505]
Best params: {'bagging_fraction': 0.8627159332342427, 'feature_fraction': 0.6389820978299352, 'lambda_l1': 5.888023333536449, 'lambda_l2': 0.1872404029272634, 'learning_rate': 0.11471367450697005, 'max_depth': 11, 'num_iterations': 92, 'num_leaves': 159, 'other_rate': 0.21200000000000002, 'seed': 0, 'top_rate': 0.09860000000000001}
Best lags: [1, 4, 7]
Other info: {'box_cox': 2.6052782418950993, 'box_cox_biasadj': False}


In [3]:
# now we can run our model with the best paramaters, best lags and other info such as box_cox and box_cox_biasadj
best_box_cox = other_["box_cox"]
best_box_cox_biasadj = other_["box_cox_biasadj"]

ml_model = ml_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_col='admissions', lags = list(best_lags), box_cox=best_box_cox, box_cox_biasadj=best_box_cox_biasadj,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)
ml_forecasts = ml_model.forecast(H=30, exog=test[cat_variables])
ml_forecasts

array([8841.81486603, 8717.4239207 , 8728.03361354, 8799.30129603,
       8783.23283169, 8829.27833182, 8831.35707669, 8773.52192027,
       8675.8802424 , 8723.57301335, 8794.69869985, 8806.26402367,
       8868.47146082, 8901.54206927, 8854.95568152, 8747.79909022,
       8722.48427292, 8802.57920304, 8809.96362276, 8837.68339764,
       8851.87532028, 8826.12552453, 8710.89686884, 8716.18233923,
       8787.31525071, 8774.0856259 , 8796.41906618, 8818.38052154,
       8778.91837493, 8667.30891302])

### optuna_tune example for univariate forecasting

In [4]:
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import MAE, RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import optuna_tune
load_wales_admissions["day_of_week"] = load_wales_admissions.index.dayofweek
load_wales_admissions["month"] = load_wales_admissions.index.month
# split the data into train and test sets
train = load_wales_admissions[:-30]
test = load_wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
ml_model = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables, categorical_encoder=ohe)
ml_model.fit(train)
# ml_model.data_prep(train)
forecasts = ml_model.forecast(H=30, exog=test[cat_variables])
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "top_rate":          lambda t: t.suggest_float("top_rate", 0.05, 0.4),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "lags":              lambda t: t.suggest_categorical(
                             "lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14],
                                 [1,2,3,4,5,6,7,14,21],
                                 [1,2,3],
                             ]),
                             
    "seed":              lambda t: 0,   # fixed, not sampled
}

best_params, best_lags, other_ = optuna_tune(
    model=ml_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=10,
    param_space=lgb_param_space, verbose=False
)
print("Best params:", best_params)
print("Best lags:", best_lags)

Best params: {'learning_rate': 0.09373778058791865, 'num_leaves': 200, 'max_depth': 8, 'bagging_fraction': 0.7836517263388818, 'feature_fraction': 0.8429125150098808, 'lambda_l2': 2.166831487616988, 'lambda_l1': 9.722399273697267, 'top_rate': 0.29911802702385065, 'other_rate': 0.14553971979590152, 'num_iterations': 73}
Best lags: [1, 2, 3, 4, 5]


### Selecting the best ETS (Error, Trend, Seasonality) model usig `optuna_tune` or `hyperopt_tune`

In [5]:
from peshbeen.models import ets

ets_param_space = {
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "trend":              lambda t: t.suggest_categorical(
                             "trend", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "seasonal":           lambda t: t.suggest_categorical(
                             "seasonal", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "smoothing_trend":    lambda t: t.suggest_float("smoothing_trend", 0.001, 0.99),
    "smoothing_seasonal": lambda t: t.suggest_float("smoothing_seasonal", 0.001, 0.99),
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "seasonal_periods":              lambda t: 7,   # fixed, not sampled
        "box_cox":           lambda t: t.suggest_float("box_cox", 0.0, 4),
        "box_cox_biasadj":   lambda t: t.suggest_categorical("box_cox_biasadj", [True, False])
}

ets_model = ets(target_col='admissions')
best_params, _, other_ = optuna_tune(
    model=ets_model,
    df=train,
    cv_split=4,
    step_size=1,
    test_size=30,
    eval_metric=RMSE,
    eval_num=100,
    param_space=ets_param_space, verbose=False
)
print("Best params:", best_params)
print("Other info:", other_)

Best params: {'smoothing_level': 0.03913300987559443, 'trend': None, 'seasonal': None, 'smoothing_trend': 0.6997912601799381, 'smoothing_seasonal': 0.30307007785543216}
Other info: {'box_cox': 1.2615377211391767, 'box_cox_biasadj': True}


In [6]:
# now forecast wit the best parameters
best_params.update(other_)
ets_model = ets(target_col='admissions', **best_params)
ets_model.fit(train)
ets_forecasts = ets_model.forecast(H=30)

In [7]:
# get cross validation results with the best parameters
cv_df = ets_model.cross_validate(
    df=train,
    cv_split=5,
        step_size=7,
        test_size=30,
        metrics=[RMSE, MAE])
cv_df.head()

,cutoff,index,split,y_true,y_pred
0,2022-12-07,2022-12-07,fold_1,8933,8919.280743
1,2022-12-07,2022-12-08,fold_1,9013,8919.280743
2,2022-12-07,2022-12-09,fold_1,9000,8919.280743
3,2022-12-07,2022-12-10,fold_1,8821,8919.280743
4,2022-12-07,2022-12-11,fold_1,8835,8919.280743


### Selecting the best orders for an ARIMA model using `optuna_tune` or `hyperopt_tune`

In [8]:
from peshbeen.models import arima
from itertools import product
# Define the hyperparameter search space for ARIMA
p_values = [0, 1, 2, 3]
d_values = [1]
q_values = [0, 1, 2, 3]

# create the list of orders using the product of p, d and q values
orders = list(product(p_values, d_values, q_values))

# Define the hyperparameter search space for seasonal ARIMA
P_values = [0, 1, 2, 3]
D_values = [0, 1]
Q_values = [0, 1, 2, 3]

# create the list of seasonal orders using the product of P, D and Q values
seasonal_orders = list(product(P_values, D_values, Q_values))

# let's define the hyperparameter space for arima using hyperopt
arima_param_space = {
    "order": hp.choice("order", orders),
    "seasonal_order": hp.choice("seasonal_order", seasonal_orders),
    "seasonal_length": 7,
    "box_cox": hp.uniform("box_cox", 0.0, 4),
    "box_cox_biasadj": hp.choice("box_cox_biasadj", [True, False])
}

arima_model = arima(target_col='admissions')
best_params, _, other_ = hyperopt_tune(
    model=arima_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=5,
    param_space=arima_param_space
)

100%|██████████| 5/5 [00:13<00:00,  2.74s/trial, best loss: 129.2170819498678]


### hyperopt_tune example for multivariate forecasting

In [9]:
from peshbeen.datasets import load_admission_calls
from peshbeen.models import ml_mv_forecaster
from peshbeen.metrics import RMSE
from peshbeen.model_selection import mv_hyperopt_tune
from lightgbm import LGBMRegressor
load_admission_calls["day_of_week"] = load_admission_calls.index.dayofweek
load_admission_calls["month"] = load_admission_calls.index.month
train = load_admission_calls[:-30]
test = load_admission_calls[-30:]

cat_variables = ["day_of_week", "month"]
mv_model = ml_mv_forecaster(model=LGBMRegressor(verbose=-1),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables,
                difference={"admissions": 1, "calls": 1}, categorical_encoder=ohe)
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
           'min_data_in_leaf': scope.int(hp.quniform ('min_data_in_leaf', 5, 100, 1)), 
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'top_k': scope.int(hp.quniform('top_k', 8, 30, 1)),
                "seed":0, 'lags': hp.choice("lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14]])}
best_params, best_lags = mv_hyperopt_tune(model=mv_model, df=train, target_col= "admissions", cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=4,
                                        param_space=lgb_param_space)
print("Best params:", best_params)
print("Best lags:", best_lags)

100%|██████████| 4/4 [00:07<00:00,  2.00s/trial, best loss: 211.15057397354335]
Best params: {'bagging_fraction': 0.6260644754229144, 'feature_fraction': 0.8967148314368943, 'lambda_l1': 4.1607495871822255, 'lambda_l2': 9.146706285919876, 'learning_rate': 0.09201129496555069, 'max_depth': 10, 'min_data_in_leaf': 61, 'num_iterations': 309, 'num_leaves': 193, 'other_rate': 0.0685, 'seed': 0, 'top_k': 25}
Best lags: {'admissions': [1, 2, 3, 4, 5], 'calls': [1, 2, 3, 4, 5]}


In [10]:
# forecast with the best parameters and best lags
mv_model = ml_mv_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_cols=['admissions', "calls"], lags = best_lags,
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})
mv_model.fit(train)
mv_forecasts = mv_model.forecast(H=30, exog=test[cat_variables])
mv_forecasts

{'admissions': array([7956.33431593, 7986.5195834 , 7986.26110118, 7964.81372194,
        7953.54168233, 7950.6169608 , 7954.82005758, 7928.31589778,
        7908.51165898, 7885.19752188, 8021.75186801, 8068.53248159,
        8102.00958928, 8039.86329994, 8051.8281892 , 7914.30315198,
        7957.54563584, 7969.02756775, 7953.11553168, 7931.97482856,
        7901.02454186, 7926.76026344, 7961.38235867, 8033.08130509,
        8070.3432    , 8040.72222709, 8074.39865454, 8023.61350946,
        8094.16754399, 8176.62393623]),
 'calls': array([1225.35588357, 1247.45142063, 1238.54681599, 1233.23638324,
        1244.01029293, 1234.94852792, 1255.27451154, 1259.04287099,
        1271.97985342, 1284.4680757 , 1276.747098  , 1261.17632091,
        1282.9527373 , 1283.75153877, 1329.74475335, 1277.55438501,
        1286.56640488, 1335.49292844, 1296.39953432, 1290.71896863,
        1270.02839287, 1360.38693673, 1321.33229693, 1378.21770604,
        1309.10290547, 1340.16502773, 1321.02862289, 

### optuna_tune example for multivariate forecasting

In [11]:
from peshbeen.model_selection import mv_optuna_tune
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "min_data_in_leaf":  lambda t: t.suggest_int("min_data_in_leaf", 5, 100),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "top_k":             lambda t: t.suggest_int("top_k", 8, 30),
    "seed":              lambda t: 0,   # fixed, not sampled
    'lags': lambda t: t.suggest_categorical(
                             "lags", [1,2,3,4,5,
                                [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14]
                             ]),    
}

mv_model = ml_mv_forecaster(model=LGBMRegressor(verbose=-1),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})

best_params, best_lags = mv_optuna_tune(model=mv_model, df=train, target_col= "admissions", cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=4,
                                        param_space=lgb_param_space)

In [12]:
# forecast with the best parameters and best lags from optuna
mv_model = ml_mv_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_cols=['admissions', "calls"], lags = best_lags,
                cat_variables=cat_variables, categorical_encoder=ohe,
                difference={"admissions": 1, "calls": 1})
mv_model.fit(train)
mv_forecasts = mv_model.forecast(H=30, exog=test[cat_variables])
mv_forecasts

{'admissions': array([8008.8862273 , 8179.87222797, 8198.42499346, 8182.71394633,
        8110.56619736, 8045.52936021, 7979.5503405 , 7994.94765268,
        8106.26826991, 8160.58230002, 8164.77228362, 8146.66307145,
        8043.63271531, 7982.53515135, 8053.28895741, 8157.75831786,
        8179.65090059, 8094.83122724, 8064.57487547, 7975.47333102,
        7815.73504871, 7921.0966348 , 8087.63080026, 8133.56379271,
        8130.43017051, 8095.13219522, 7973.47467997, 7814.88270269,
        7882.02517028, 8060.11206863]),
 'calls': array([1227.87790522, 1257.32352163, 1230.22401898, 1247.24818067,
        1197.05046854, 1286.05587055, 1295.0549482 , 1237.75583185,
        1317.6794118 , 1261.74156945, 1268.41832939, 1274.28053428,
        1311.15200853, 1318.11696802, 1288.55919003, 1351.9154056 ,
        1292.67286827, 1326.90342246, 1287.08937061, 1364.10522781,
        1340.62171947, 1341.43341055, 1351.79008315, 1302.55868857,
        1278.00968252, 1272.33082072, 1337.13800522, 